MOCK DATASET 
- subset of full dataset for pipeline validation 
- API request --> flat parquet

In [ ]:
import requests
import json

# Scarica il manifest completo
manifest_url = "https://api.fda.gov/download.json"
manifest = requests.get(manifest_url).json()

# Estrai solo i file drug/event
all_partitions = manifest["results"]["drug"]["event"]["partitions"]

print(f"Totale file disponibili: {len(all_partitions)}")
print(f"Esempio struttura:\n{json.dumps(all_partitions[0], indent=2)}")

In [ ]:
# Quarter strategici scelti per coprire tutte le ere e i fenomeni di bias
TARGET_QUARTERS = {
    "2004q1",  # era AERS legacy (formato più vecchio)
    "2007q1",  # approvazione lapatinib - inizio Weber effect
    "2008q4",  # fine anno 1 post-approvazione
    "2012q1",  # transizione strutturale FAERS
    "2015q2",  # metà periodo, segnale consolidato
    "2019q1",  # formato moderno
    "2022q4",  # anni recenti
    "2024q2",  # più recente disponibile
}

# Prendi SOLO la prima parte di ogni quarter (campione rappresentativo)
selected_files = []
for p in all_partitions:
    file_url = p["file"]
    # Estrai il quarter dalla URL: .../drug/event/2007q1/drug-event-0001-of-...
    parts = file_url.split("/")
    quarter = parts[-2]  # es. "2007q1"
    filename = parts[-1]  # es. "drug-event-0001-of-0003.json.zip"
    
    if quarter in TARGET_QUARTERS and "0001-of" in filename:
        selected_files.append({
            "quarter": quarter,
            "file": file_url,
            "size_mb": float(p["size_mb"]),
            "records": p["records"]
        })

# Mostra il piano di download
import pandas as pd
df_plan = pd.DataFrame(selected_files).sort_values("quarter")
print(df_plan.to_string(index=False))
print(f"\nDimensione totale stimata: {df_plan['size_mb'].sum():.1f} MB zippati")
print(f"Record totali stimati: {df_plan['records'].sum():,}")

In [ ]:
import requests
import zipfile
import io
from pathlib import Path

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

def download_and_extract(entry):
    url = entry["file"]
    quarter = entry["quarter"]
    out_path = RAW_DIR / f"{quarter}.json"
    
    if out_path.exists():
        print(f"  [SKIP] {quarter} già presente")
        return out_path
    
    print(f"  [DOWNLOAD] {quarter} ({entry['size_mb']} MB zip, ~{entry['records']:,} records)...")
    
    try:
        # verify=False: accettabile per download pubblici FDA,
        # ma sopprime il warning per non intasare l'output
        import urllib3
        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
        
        response = requests.get(url, verify=False, timeout=300, stream=True)
        response.raise_for_status()
        
        zip_bytes = io.BytesIO(response.content)
        
        with zipfile.ZipFile(zip_bytes) as zf:
            json_filename = [n for n in zf.namelist() if n.endswith(".json")][0]
            json_bytes = zf.read(json_filename)
        
        out_path.write_bytes(json_bytes)
        print(f"  [OK] {out_path} ({out_path.stat().st_size / 1e6:.1f} MB decompressi)")
        return out_path
        
    except Exception as e:
        print(f"  [ERROR] {quarter}: {e}")
        return None

# Riesegui il download
downloaded = []
for entry in selected_files:
    path = download_and_extract(entry)
    if path:
        downloaded.append({"quarter": entry["quarter"], "path": str(path)})

print(f"\nFile scaricati: {len(downloaded)}/{len(selected_files)}")

In [ ]:
import json
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

PARQUET_PATH = Path("data/faers_flat.parquet")

# ── Funzioni di normalizzazione ───────────────────────────────────────────────

def parse_age_to_years(age_val, age_unit_code):
    if age_val is None:
        return None
    try:
        val = float(str(age_val).strip())
    except (ValueError, TypeError):
        return None
    unit = str(age_unit_code).strip() if age_unit_code else "801"
    conversions = {"801": 1.0, "802": 1/12, "803": 1/52.18,
                   "804": 1/365.25, "805": 1/8766}
    return val * conversions.get(unit, 1.0)

def age_to_stratum(age_years):
    """
    Converte età in anni nel gruppo demografico per la stratificazione
    della contingency table:
        pediatric  → < 18 anni
        adult      → 18–64 anni
        geriatric  → ≥ 65 anni
        unknown    → dato mancante

    Valori usati come filtro DuckDB:
        WHERE age_stratum = 'pediatric'
    """
    if age_years is None:
        return "unknown"
    if age_years < 18:
        return "pediatric"
    if age_years < 65:
        return "adult"
    return "geriatric"

def parse_sex(sex_val):
    """
    Normalizza il campo sesso per la stratificazione della contingency table:
        male / female / unknown

    Valori usati come filtro DuckDB:
        WHERE sex = 'female'
    """
    if sex_val is None:
        return "unknown"
    s = str(sex_val).strip().upper()
    if s in ("1", "M", "MALE"):
        return "male"
    if s in ("2", "F", "FEMALE"):
        return "female"
    return "unknown"

def parse_date(date_str):
    if not date_str:
        return None
    s = str(date_str).strip().replace("-", "")
    try:
        if len(s) == 8:
            return f"{s[:4]}-{s[4:6]}-{s[6:8]}"
        if len(s) == 6:
            return f"{s[:4]}-{s[4:6]}-01"
        if len(s) == 4:
            return f"{s}-01-01"
    except Exception:
        pass
    return None

def extract_drug_name_with_source(drug: dict) -> tuple:
    """
    Fallback chain per il nome del farmaco:
    1. activesubstance.activesubstancename  (era moderna 2019+)
    2. medicinalproduct                     (tutte le ere, nome commerciale)
    3. openfda.generic_name[0]             (era legacy 2004-2012)
    4. openfda.brand_name[0]              (fallback commerciale legacy)
    5. openfda.substance_name[0]          (ultimo tentativo)
    Restituisce (nome_normalizzato, fonte) per tracciabilità QC.
    """
    active = drug.get("activesubstance") or {}
    if isinstance(active, dict):
        name = str(active.get("activesubstancename") or "").upper().strip()
        if name:
            return name, "activesubstance"

    name = str(drug.get("medicinalproduct") or "").upper().strip()
    if name:
        return name, "medicinalproduct"

    openfda = drug.get("openfda") or {}
    if isinstance(openfda, dict):
        for field in ("generic_name", "brand_name", "substance_name"):
            val = openfda.get(field)
            if val:
                if isinstance(val, list) and val:
                    name = str(val[0]).upper().strip()
                elif isinstance(val, str):
                    name = val.upper().strip()
                if name:
                    return name, f"openfda.{field}"

    return "", "missing"

def flatten_report(report):
    """
    Appiattisce un singolo report FAERS nel formato:
        una riga per ogni combinazione (drug × reaction)

    Ogni coppia (drug, reaction) è l'unità statistica di base per la
    disproportionality analysis.

    Le comedications NON sono serializzate come colonna separata: il self-join
    su safetyreportid nel Parquet permette a build_contingency_tables_by_comed()
    di recuperare tutti i drug co-presenti in un report senza ridondanza nei dati.

    Colonne per la stratificazione demografica della contingency table:
        age_stratum  →  pediatric / adult / geriatric / unknown
        sex          →  male / female / unknown
    Passate come filtro WHERE alla query DuckDB:
        build_contingency_tables_duckdb(..., filters="sex = 'female'")
        build_contingency_tables_duckdb(..., filters="age_stratum = 'geriatric'")
    """
    rows = []

    # ── Campi report-level ───────────────────────────────────────────────────
    report_id    = report.get("safetyreportid")
    receive_date = parse_date(report.get("receivedate") or report.get("receiptdate"))
    serious      = report.get("serious")

    outcome_death = report.get("seriousnessdeath", "0")
    outcome_lt    = report.get("seriousnesslifethreatening", "0")
    outcome_hosp  = report.get("seriousnesshospitalization", "0")
    outcome_disab = report.get("seriousnessdisabling", "0")

    primary_source   = report.get("primarysource") or {}
    reporter_country = primary_source.get("reportercountry")
    reporter_qual    = primary_source.get("qualification")

    # ── Paziente ─────────────────────────────────────────────────────────────
    patient     = report.get("patient") or {}
    age_years   = parse_age_to_years(
                      patient.get("patientonsetage"),
                      patient.get("patientonsetageunit"))
    age_stratum = age_to_stratum(age_years)
    sex         = parse_sex(patient.get("patientsex"))

    # ── Farmaci ──────────────────────────────────────────────────────────────
    drugs = patient.get("drug", [])
    if not isinstance(drugs, list):
        drugs = [drugs] if drugs else []

    # ── Reazioni ─────────────────────────────────────────────────────────────
    reactions = patient.get("reaction", [])
    if not isinstance(reactions, list):
        reactions = [reactions] if reactions else []

    reaction_pts = [
        r.get("reactionmeddrapt", "").upper().strip()
        for r in reactions
        if isinstance(r, dict) and r.get("reactionmeddrapt")
    ]

    # ── Cross join drug × reaction ────────────────────────────────────────────
    # Ogni riga rappresenta una coppia (drug, PT) osservata in questo report.
    # Le comedications si ricavano a posteriori con self-join su safetyreportid:
    #   SELECT DISTINCT safetyreportid, drug_name AS comedication
    #   FROM parquet WHERE drug_name != <target>
    for drug in drugs:
        if not isinstance(drug, dict):
            continue

        char                        = str(drug.get("drugcharacterization", "") or "").strip()
        drug_name, drug_name_source = extract_drug_name_with_source(drug)
        indication                  = str(drug.get("drugindication") or "").upper().strip()

        for reaction_pt in reaction_pts:
            rows.append({
                "safetyreportid":        str(report_id) if report_id else None,
                "receivedate":           receive_date,
                "receive_year":          int(receive_date[:4]) if receive_date else None,
                "receive_quarter":       (
                    f"{receive_date[:4]}Q{(int(receive_date[5:7])-1)//3+1}"
                    if receive_date and len(receive_date) >= 7 else None
                ),
                "serious":               str(serious) if serious else None,
                "outcome_death":         str(outcome_death),
                "outcome_lifethreat":    str(outcome_lt),
                "outcome_hosp":          str(outcome_hosp),
                "outcome_disab":         str(outcome_disab),
                "reporter_country":      reporter_country,
                "reporter_qual":         str(reporter_qual) if reporter_qual else None,
                "age_years":             age_years,
                "age_stratum":           age_stratum,
                "sex":                   sex,
                "drug_name":             drug_name,
                "drug_name_source":      drug_name_source,
                "drug_characterization": char,
                "drug_indication":       indication if indication else None,
                "reaction_pt":           reaction_pt,
            })

    return rows

# ── Flattening di tutti i file scaricati → Parquet ───────────────────────────

all_rows = []
stats    = {}

for entry in downloaded:
    quarter = entry["quarter"]
    path    = Path(entry["path"])
    print(f"Parsing {quarter}...", end=" ")

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    reports      = data.get("results", [])
    quarter_rows = []

    for report in reports:
        quarter_rows.extend(flatten_report(report))

    stats[quarter] = {
        "n_reports":   len(reports),
        "n_rows_flat": len(quarter_rows)
    }
    all_rows.extend(quarter_rows)
    print(f"{len(reports):,} reports → {len(quarter_rows):,} righe")

# ── Salva Parquet ─────────────────────────────────────────────────────────────

df = pd.DataFrame(all_rows)

print(f"\nSchema finale:\n{df.dtypes}")
print(f"Totale righe: {len(df):,}")
print(f"Valori nulli per colonna:\n{df.isnull().sum()}")

df.to_parquet(PARQUET_PATH, index=False, engine="pyarrow")
print(f"\nSalvato: {PARQUET_PATH} ({PARQUET_PATH.stat().st_size / 1e6:.1f} MB)")

# ── Verifica immediata drug_name_source ───────────────────────────────────────

import duckdb
con = duckdb.connect()
P   = str(PARQUET_PATH)

print("\n=== Distribuzione drug_name_source ===")
print(con.execute(f"""
    SELECT drug_name_source,
           COUNT(*) as n_rows,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) as pct
    FROM '{P}'
    GROUP BY drug_name_source
    ORDER BY n_rows DESC
""").df().to_string(index=False))

print("\n=== Missing drug_name residuale per quarter ===")
print(con.execute(f"""
    SELECT receive_quarter,
           COUNT(*) as total,
           SUM(CASE WHEN drug_name = '' OR drug_name IS NULL THEN 1 ELSE 0 END) as missing,
           ROUND(100.0 * SUM(CASE WHEN drug_name = '' OR drug_name IS NULL
                             THEN 1 ELSE 0 END) / COUNT(*), 1) as pct_missing
    FROM '{P}'
    GROUP BY receive_quarter
    HAVING SUM(CASE WHEN drug_name = '' OR drug_name IS NULL THEN 1 ELSE 0 END) > 0
       AND ROUND(100.0 * SUM(CASE WHEN drug_name = '' OR drug_name IS NULL
                             THEN 1 ELSE 0 END) / COUNT(*), 1) > 5
    ORDER BY pct_missing DESC
""").df().to_string(index=False))


In [ ]:
import duckdb

con = duckdb.connect()
P   = str(PARQUET_PATH)

# ── Schema ───────────────────────────────────────────────────────────────────
print("=== Schema ===")
print(con.execute(f"DESCRIBE SELECT * FROM '{P}'").df().to_string(index=False))

# ── Distribuzione temporale (Weber effect check) ──────────────────────────────
print("\n=== Report per anno ===")
print(con.execute(f"""
    SELECT receive_year, COUNT(DISTINCT safetyreportid) AS n_reports
    FROM '{P}'
    WHERE safetyreportid IS NOT NULL
    GROUP BY receive_year
    ORDER BY receive_year
""").df().to_string(index=False))

# ── Top farmaci (primary suspect) ────────────────────────────────────────────
print("\n=== Top farmaci (primary suspect) ===")
print(con.execute(f"""
    SELECT drug_name, COUNT(*) AS n
    FROM '{P}'
    WHERE drug_characterization = '1'
    GROUP BY drug_name
    ORDER BY n DESC
    LIMIT 20
""").df().to_string(index=False))

# ── Stratificazione demografica ───────────────────────────────────────────────
print("\n=== Distribuzione sex ===")
print(con.execute(f"""
    SELECT sex,
           COUNT(DISTINCT safetyreportid) AS n_reports,
           ROUND(100.0 * COUNT(DISTINCT safetyreportid) /
                 SUM(COUNT(DISTINCT safetyreportid)) OVER (), 1) AS pct
    FROM '{P}'
    GROUP BY sex
    ORDER BY n_reports DESC
""").df().to_string(index=False))

print("\n=== Distribuzione age_stratum ===")
print(con.execute(f"""
    SELECT age_stratum,
           COUNT(DISTINCT safetyreportid) AS n_reports,
           ROUND(100.0 * COUNT(DISTINCT safetyreportid) /
                 SUM(COUNT(DISTINCT safetyreportid)) OVER (), 1) AS pct
    FROM '{P}'
    GROUP BY age_stratum
    ORDER BY n_reports DESC
""").df().to_string(index=False))

# ── Copertura comedications via self-join ─────────────────────────────────────
# Le comedications non sono una colonna esplicita: si ricavano dal self-join
# su safetyreportid. Verifichiamo qui quanti report hanno almeno 2 drug distinti
# (condizione necessaria per avere una comedication).
print("\n=== Report con almeno 1 comedication (n drug distinti per report) ===")
print(con.execute(f"""
    SELECT
        COUNT(DISTINCT safetyreportid)                             AS total_reports,
        SUM(CASE WHEN n_drugs > 1 THEN 1 ELSE 0 END)              AS reports_with_comed,
        ROUND(100.0 * SUM(CASE WHEN n_drugs > 1 THEN 1 ELSE 0 END)
              / COUNT(DISTINCT safetyreportid), 1)                 AS pct_with_comed
    FROM (
        SELECT safetyreportid, COUNT(DISTINCT drug_name) AS n_drugs
        FROM '{P}'
        WHERE drug_name IS NOT NULL AND drug_name != ''
        GROUP BY safetyreportid
    )
""").df().to_string(index=False))

# ── Reminder: API delle funzioni contingency table ────────────────────────────
print("""
=== REMINDER: build_contingency_tables_duckdb() ===

  Analisi globale:
    ct = build_contingency_tables_duckdb(PARQUET_PATH, target_drug="LAPATINIB")

  Stratificata per sesso:
    ct = build_contingency_tables_duckdb(PARQUET_PATH, target_drug="LAPATINIB",
                                          extra_filters="sex = 'female'")

  Stratificata per età:
    ct = build_contingency_tables_duckdb(PARQUET_PATH, target_drug="LAPATINIB",
                                          extra_filters="age_stratum = 'geriatric'")

  Combinata:
    ct = build_contingency_tables_duckdb(PARQUET_PATH, target_drug="LAPATINIB",
                                          extra_filters="sex = 'female'
                                                         AND age_stratum = 'adult'")

=== REMINDER: build_contingency_tables_by_comed() ===

  Segnali di lapatinib stratificati per OGNI comedication presente nei report:
    ct = build_contingency_tables_by_comed(PARQUET_PATH, target_drug="LAPATINIB")

  Con filtro demografico aggiuntivo:
    ct = build_contingency_tables_by_comed(PARQUET_PATH, target_drug="LAPATINIB",
                                            extra_filters="sex = 'female'")

  Output: (drug, pt, comedication, a, b, c, d, n) — una riga per coppia (PT, comedication)
  Passato direttamente a compute_ror() / compute_prr() / compute_bcpnn() / compute_ebgm()
""")
